In [ ]:
import sys
sys.path.append("..")

import glob
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms as T

from PIL import Image
from collections import deque
from torchmetrics import Metric

from src.metrics import SegIoU, WarpedTC
from src.utils import warp


In [ ]:
p1 = "/data/video-segmentation/cityscapes/leftImg8bit_sequence/train/aachen/aachen_000000_000019_leftImg8bit.png"
p2 = "/data/video-segmentation/cityscapes/leftImg8bit_sequence/train/aachen/aachen_000000_000023_leftImg8bit.png"
label_path = "/data/video-segmentation/cityscapes/gtFine/train/aachen/aachen_000000_000019_gtFine_labelTrainIds.png"


img1 = Image.open(p1)
img2 = Image.open(p2)
img1

In [ ]:
test1 = torch.randint(0, 5, (3, 17, 23))
test2 = torch.randint(0, 5, (3, 17, 23))

test1[0, 0, 0] = 255

In [ ]:
seg_iou = SegIoU(5, 255)
seg_iou.update(test2, test1)
seg_iou.compute()

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

weights = Raft_Large_Weights.DEFAULT
raft = raft_large(weights=weights).eval().cuda()
raft_tf = weights.transforms()

@torch.inference_mode()
def compute_flow(src, dst):
    """Flow src -> dst. src, dst: (B,3,H,W) float in [0,1]; H, W divisible by 8."""
    a, b = raft_tf(src, dst)
    return raft(a, b)[-1]

a, b = raft_tf(img1.reduce(2), img2.reduce(2))
a = a.unsqueeze(0).to("cuda")
b = b.unsqueeze(0).to("cuda")

# size = 128
# y, x = torch.meshgrid(torch.arange(size), torch.arange(size))
# pos1 = (torch.stack((x, y), dim=-1) - size / 2) / torch.tensor(4)
# pos2 = (torch.stack((x, y), dim=-1) - size / 2) / torch.tensor(2)
# a = torch.exp(-0.5 * torch.square(pos1).sum(-1))
# b = torch.exp(-0.5 * torch.square(pos2).sum(-1))
# a = a[None, None, :].expand(-1, 3, -1, -1).to("cuda")
# b = b[None, None, :].expand(-1, 3, -1, -1).to("cuda")

with torch.no_grad():
    flow_fw = raft(a, b)[-1]
    flow_bw = raft(b, a)[-1]

In [ ]:
pred_t = torch.randint(0, 5, b[:, 0].shape).to("cuda")
pred_prev = torch.randint(0, 5, a[:, 0].shape).to("cuda")

warped_tc = WarpedTC(5).to("cuda")
warped_tc.update(pred_t, pred_prev, flow_bw, flow_fw)
warped_tc.compute()

In [ ]:
# tc = WarpedTC().cuda()

# @torch.inference_mode()
# def eval_snippet(frames):  # frames: (T,3,H,W) float in [0,1] on cuda
#     preds = torch.cat([to_pred(model(f[None]), f.shape[-2:]) for f in frames])
#     for t in range(1, len(frames)):
#         prev, cur = frames[t - 1 : t], frames[t : t + 1]
#         flow_bw = compute_flow(cur, prev)   # t -> t-1
#         flow_fw = compute_flow(prev, cur)   # t-1 -> t
#         tc.update(preds[t : t + 1], preds[t - 1 : t], flow_bw, flow_fw)

# # for frames in snippets: eval_snippet(frames)
# # print(tc.compute())

# torch.manual_seed(0)
# B, H, W = 1, 128, 256
# p = F.interpolate(torch.randint(0, 19, (B, 1, 8, 16)).float(), size=(H, W)).squeeze(1).long()
# prev = torch.roll(p, 4, dims=-1)                      # objects were 4 px to the right at t-1
# flow_bw = torch.zeros(B, 2, H, W); flow_bw[:, 0] = 4  # t -> t-1: +4 in x
# flow_fw = -flow_bw

# m = WarpedTC(); m.update(p, prev, flow_bw, flow_fw)
# print("correct flow:", m.compute())                   # TC = 1.0, coverage < 1 (right edge masked)

# m = WarpedTC(); z = torch.zeros_like(flow_bw); m.update(p, prev, z, z)
# print("zero flow:   ", m.compute())                   # TC < 1

In [ ]:

class VEC(Metric):
    """Video Effective Consistency (DTERN, ICCV 2025), sliding window, accumulated over the dataset.

    Over each window of `window` consecutive frames, a pixel is stable in the ground truth (or prediction) if
    its class never changes; per class, VEC is the IoU between the stable-and-correct pixels. Needs labels on
    every frame, so it applies to KITTI-STEP, not Cityscapes (one labeled frame per snippet).

    Call update(pred, target) once per frame, in order, and new_sequence() before each new sequence. Use
    update(), not metric(...): the window lives outside the metric state, and Metric.forward resets state
    around each call. Under DDP, each sequence must be evaluated whole on one process.
    """

    is_differentiable = False
    higher_is_better = True
    full_state_update = False

    def __init__(self, num_classes: int = 19, window: int = 8, ignore_index: int = 255, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.window = window
        self.ignore_index = ignore_index

        self.add_state("inter", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("gt_cnt", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("pr_cnt", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self._buf = deque(maxlen=window)

    def forward(self, *args, **kwargs):
        raise RuntimeError("VEC keeps a sliding window across calls; use update(), not metric(...)")

    def new_sequence(self) -> None:
        self._buf.clear()

    def update(self, pred: Tensor, target: Tensor) -> None:
        """pred, target: (H, W) class ids for one frame."""
        self._buf.append((pred, target))
        if len(self._buf) < self.window:
            return

        preds = torch.stack([p for p, _ in self._buf])  # (n, H, W)
        gts = torch.stack([g for _, g in self._buf])
        g0, p0 = gts[0], preds[0]

        valid = (gts != self.ignore_index).all(0)
        gt_stable = (gts == g0).all(0) & valid
        pr_stable = (preds == p0).all(0) & valid
        both = gt_stable & pr_stable & (g0 == p0)

        C = self.num_classes
        self.inter += torch.bincount(g0[both], minlength=C)[:C]
        self.gt_cnt += torch.bincount(g0[gt_stable], minlength=C)[:C]
        self.pr_cnt += torch.bincount(p0[pr_stable], minlength=C)[:C]

    def compute(self) -> dict[str, Tensor]:
        inter = self.inter.double()
        union = self.gt_cnt.double() + self.pr_cnt.double() - inter
        iou = inter / union  # 0/0 -> nan for absent classes

        return {"VEC": iou.nanmean(), "VEC_per_class": iou}

    def reset(self) -> None:
        super().reset()
        self._buf.clear()


In [ ]:

class VEC(Metric):
    """Video Effective Consistency (DTERN, ICCV 2025), sliding window, dataset-level accumulation.

    Call update(pred, target) once per frame, in order. Call new_sequence() before each new video.
    """
    is_differentiable = False
    higher_is_better = True
    full_state_update = False

    def __init__(self, num_classes=19, window=8, ignore_index=255, **kwargs):
        super().__init__(**kwargs)
        self.C, self.n, self.ignore_index = num_classes, window, ignore_index
        self.add_state("inter", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("gt_cnt", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self.add_state("pr_cnt", torch.zeros(num_classes, dtype=torch.long), dist_reduce_fx="sum")
        self._buf = deque(maxlen=window)

    def new_sequence(self):
        self._buf.clear()

    def update(self, pred, target):
        """pred, target: (H,W) class ids for one frame."""
        self._buf.append((pred, target))
        if len(self._buf) < self.n:
            return

        preds = torch.stack([p for p, _ in self._buf])    # (n,H,W)
        gts = torch.stack([g for _, g in self._buf])
        g0, p0 = gts[0], preds[0]

        valid = (gts != self.ignore_index).all(0)
        gt_stable = (gts == g0).all(0) & valid
        pr_stable = (preds == p0).all(0) & valid
        both = gt_stable & pr_stable & (g0 == p0)

        C = self.C
        self.inter += torch.bincount(g0[both], minlength=C)[:C]
        self.gt_cnt += torch.bincount(g0[gt_stable], minlength=C)[:C]
        self.pr_cnt += torch.bincount(p0[pr_stable], minlength=C)[:C]

    def compute(self):
        inter = self.inter.double()
        union = self.gt_cnt.double() + self.pr_cnt.double() - inter
        iou = inter / union  # 0/0 -> nan for absent classes
        return {"VEC": iou.nanmean(), "VEC_per_class": iou}

    def reset(self):
        super().reset()
        self._buf.clear()